# 72 DPO from the current model on the new pairs (A100)

Method "cost first, then accuracy": (1) a cost-guided search finds the cheapest correct query known for every train question; (2) the current model (`rl_v1_it8`) answers every train question 8 times; (3) preference pairs: chosen = the cheap query, rejected = the model's own costlier or wrong answer; (4) DPO from the current model; (5) RL from the DPO model, with the cheapest known query as the cost yardstick; (6) the model is chosen on validation by a fixed rule, then dev and mini-dev are scored once.

Step 4. DPO with an NLL term on the chosen answer, starting from `rl_v1_it8` (which is also the reference model), the strong setting of the first DPO (learning rate 5e-5), 2 epochs. The model after epoch 1 is kept as its own run `dpo_v2_e1`; validation decides between the two. Then validation, dev and mini-dev are predicted for both (a few minutes each).

**Runtime:** A100, about 1.5 to 2 hours (depends on the number of pairs). **Resumable:** after a disconnect, run the setup and config cells (and the staging cell if there is one) again, then the cell that was running: finished work is kept.

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
EXPORT = "sft_v1"                   # SFT export: its 4 held-out train databases are the validation set
CURRENT = "rl_v1_it8"               # our current model (RL, kept on validation): the start and the reference
SEARCH = "search_v1"                # 1. cost-guided search, all train questions -> cheap targets
OWN = "own_v1"                      # 2. the current model's own answers, 8 per train question
PAIRS = "pairs_v2"                  # 3. preference pairs (main)
DPO = "dpo_v2"                      # 4. DPO from CURRENT, 2 epochs: dpo_v2_e1 (after epoch 1) and dpo_v2
DPO_RUNS = [f"{DPO}_e1", DPO]
RL = "rl_v2"                        # 5. RL from the DPO model kept on validation: rl_v2_it8 and rl_v2
RL_RUNS = [f"{RL}_it8", RL]
# Ablations (DPO only, same settings as the kept DPO model): where the chosen and rejected answers come from.
ABLATIONS = {"gold": "--chosen gold",      # chosen = gold: the generated queries are not used at all
             "pool": "--chosen pool",      # chosen = the first pool's cheapest query: no search
             "off": "--rejected pool"}     # rejected = other models' pool queries, not the model's own
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
VAL_DBS = ",".join(json.load(open(f"{R}/{EXPORT}/export_meta.json"))["holdout_db_ids"])
def done(run):
    return os.path.exists(f"{R}/{run}/adapter_meta.json")
print("validation databases:", VAL_DBS)
print(json.load(open(f"{R}/{PAIRS}/export_meta.json"))["stats"]["dpo_train"]["by_type"])
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

### 1. DPO (2 epochs)

In [ ]:
!python -m effsql.train dpo --export {PAIRS} --run {DPO} --init-from {CURRENT} --lr 5e-5 --epochs 2

### 2. Predict validation, dev and mini-dev (after epoch 1 and after epoch 2)

In [ ]:
for r in DPO_RUNS:
    !python -m effsql.train predict --run {r} --splits val,dev,minidev

### Done: disconnect now, then run `73_dpo_check` on a CPU runtime